# 01 · Baseline und erste Hypothesen (Phase 1, vorläufig) – ⚠️ AUSLAUFMODELL

> **Auslaufmodell (D-037, 14.10.2026):** Dieses Notebook gehört zu Simons Run 1 und baut auf Simons Pipeline und den
> Hypothesen H1–H3 auf (D-026), **nicht auf der Zahlenbasis des Projekts** (Ralfs Rechenkern, D-028). Es bleibt als
> Archiv lauffähig, wird aber nicht weiterentwickelt. Phase-1-Ergebnisse: `outputs/run3_kontext-v5_2026-10-14/`.
> Für Phase 2 entsteht eine neue Analyse-Werkbank auf Ralfs Kern (siehe `notebooks/README.md`).

**Frage:** Wie hängen die drei Datenbasen – Ticketing, Zutritt, Catering – auf Spielebene zusammen, und welche
Rolle spielt der Spielkontext? Das Notebook ist der Startpunkt für die Weiterarbeit an den Hypothesen.

**Datenbasis – zwei Modi (automatisch gewählt):**
- **Voll** (mit Rohdaten): `config/project.local.yaml` ist vorhanden → bereinigte Tabellen aus `data/clean/LATEST`
  (erzeugt mit `python -m fcsg.cli prepare-data`), Spieltabelle wird neu gebaut. Alle Abschnitte laufen.
- **Repo** (ohne Rohdaten): Simons Spieltabelle `mart_match.csv` wird aus dem neuesten Lauf in `outputs/run*/` geladen
  (ab Run 3 im Unterordner `simon_baseline/`). Zahlenbasis des Projekts ist Ralfs Kern (D-028, `ralf_kern_phase1/`).
  Abdeckung, Baseline, Reto-Abgleich, H1, H2, K1 und eigene Tests auf Spielebene laufen. H3 zeigt nur das gespeicherte
  Ergebnis, weil dafür die Ticket-Vorlauftabellen nötig sind.
Erzwingen: Umgebungsvariable `FCSG_NOTEBOOK_MODE=repo` bzw. `voll`.

**Methode:** Alle Rechenschritte stehen im Paket `fcsg` (`baseline.py`, `hypotheses.py`) – das Notebook ruft nur auf
und zeigt an. Zusammenhänge saison-zentriert, Spearman bzw. log-log-Steigung, Bootstrap-Konfidenzintervalle.

**Grenzen:** Definitionen SOLL/IST/Public Netto und Umsatzbasis offen (D-004/D-005); Assoziationen, keine Ursachen;
keine Potenziale; Deckungsbeitrag mangels Kostendaten nicht berechenbar (D-003). Details: `docs/analysis/README.md`.

> Ausgaben vor jedem Commit leeren (VS Code: «Clear All Outputs»).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fcsg.hypotheses as hyp
from fcsg.baseline import (add_real_kpis, build_mart_match_real, coverage_real, load_clean, reto_comparison,
                           season_summary_real)
from fcsg.config import PREPARE_MODES, load_config

pd.set_option("display.max_columns", 40)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOCAL_CFG = ROOT / "config" / "project.local.yaml"
MODE = os.environ.get("FCSG_NOTEBOOK_MODE") or ("voll" if LOCAL_CFG.exists() else "repo")
# neuester gespeicherter Lauf mit Simons Spieltabelle (Run 1: direkt im Laufordner, ab Run 3: simon_baseline/)
RUN_DIR = sorted(p.parent for p in (ROOT / "outputs").glob("run*_*/**/mart_match.csv")
                 if (p.parent / "h3_leadtime.csv").exists())[-1]

if MODE == "voll":
    cfg = load_config(LOCAL_CFG, allowed_modes=PREPARE_MODES)
    clean_id, t, clean_manifest = load_clean(cfg.clean_dir)
    res = build_mart_match_real(t)
    m = add_real_kpis(res.mart)
    checks = pd.DataFrame(res.checks)
    print(f"Modus VOLL · bereinigter Lauf {clean_id} · {len(m)} Spiele")
else:
    t = None
    m = pd.read_csv(RUN_DIR / "mart_match.csv")
    checks = pd.read_csv(RUN_DIR / "quality_checks.csv")
    print(f"Modus REPO · Spieltabelle aus {RUN_DIR.name} · {len(m)} Spiele")
checks

## 1 Abdeckung – welche Spiele gehen wo ein?

In [ ]:
coverage_real(m)

## 2 Baseline je Saison (gewichtet Σ/Σ)

In [ ]:
season_summary_real(m).round(3)

## 3 Abgleich mit Retos Sichtung

In [ ]:
h1 = hyp.test_h1(m)
reto_comparison(m, elasticity=h1.estimate).round(3)

## 4 H1 · Ticketing × Catering – wächst der Umsatz überproportional mit den Zuschauern?

In [ ]:
print(h1.status, '|', h1.interpretation)
d = m[(m["ist"] > 0) & (m["revenue_gross_chf"] > 0)].copy()
for c, src in (("x", "ist"), ("y", "revenue_gross_chf")):
    d[c] = np.log(d[src]) - np.log(d[src]).groupby(d["season"]).transform("mean")
ax = d.plot.scatter("x", "y", figsize=(5, 4), title="log Umsatz vs. log IST (saison-zentriert)")
xs = np.linspace(d.x.min(), d.x.max(), 20); ax.plot(xs, h1.estimate * xs, color="black")
h1.table

## 5 H2 · Zutritt × Catering – mehr Bons, wenn früher angekommen wird?

In [ ]:
h2 = hyp.test_h2(m)
print(h2.status, '|', h2.interpretation)
h2.table.round(3)

## 6 H3 · Ticketing × Zutritt – werden kurzfristig gekaufte Tickets häufiger genutzt?

In [ ]:
if t is not None:
    h3 = hyp.test_h3(t, m)
    print(h3.status, '|', h3.interpretation)
    h3_table = h3.table
else:
    print("Modus REPO: gespeichertes Ergebnis aus", RUN_DIR.name, "(Neuberechnung braucht data/clean)")
    h3_table = pd.read_csv(RUN_DIR / "h3_leadtime.csv")
h3_table.round(3)

## 7 K1 · Spielkontext – welche Kontextvariablen hängen mit Anwesenheit, IST und KF zusammen?

Hinweis: Hier 500 Bootstrap-Ziehungen (schneller), im Bericht 1000. Zusammenhänge, deren Intervall knapp an 0 liegt
(z. B. Wochenende, Free-TV), können deshalb zwischen «KI ohne 0» und «unklar» wechseln. Solche Grenzfälle nicht überinterpretieren.

In [ ]:
k1 = hyp.context_associations(m, n_boot=500)
print(k1.interpretation)
k1.table[k1.table["ci_excludes_0"]].round(3)

## 8 Eigene Hypothese testen (Vorlage)

Beispiel H7 aus `docs/analysis/README.md`: *Spiele mit mehr Scans nach Anpfiff haben einen tieferen Ø Bon.*
Spalten der Spieltabelle: `m.columns`; Definitionen: `fcsg.baseline.KPI_DEFINITIONS_REAL`.

In [ ]:
d = m[(m["access_complete"] == 1) & m["avg_receipt_gross_chf"].notna()].copy()
x, y = hyp.demean(d, "share_after_kickoff"), hyp.demean(d, "avg_receipt_gross_chf")
rho = hyp.spearman(x, y)
lo, hi = hyp.bootstrap(pd.DataFrame({"x": x, "y": y}), lambda s: hyp.spearman(s["x"], s["y"]))
print(f"n = {len(d)}, ρ = {rho:.2f} (95 %-KI {lo:.2f} bis {hi:.2f}) → {hyp.status_from_ci(lo, hi, 0.0, expected_sign=-1)}")

## 9 Verprobung und offene Entscheidungen
- Stichprobe: 2–3 Spiele gegen die Quell-Excel prüfen (IST, Catering-Umsatz, Transaktionen).
- Prüfprotokoll oben (Zelle 1) muss überall OK sein; WARN-Einträge im Bericht erklärt.
- Offen: D-003 (Kostendaten für Deckungsbeitrag), D-004/D-005 (Definitionen), D-023 (Datenraum), Fragen an Sandro.